In [ ]:
"""
tilting_computation_gain_benchmark.py

Compares online encrypted computation time for:

  Case 1 (tilted, tilde_kappa):
    Cloud computes h_ell(g(U)) only -- mirrors _worker_run_cycle in online.py.
    Online ops: EvalAdd (for residual) + Horner polynomial evaluation.

  Case 2 (no tilting, kappa_0):
    Cloud computes h_ell(g(U)) AND the quadratic cost J_Q(U), where:
        J_Q(U) = (S^T x_0)^T U + (1/2) U^T H U
    S, H, x_0, U are all private (encrypted); all products are ct-ct.

For Case 2, surrogate and J_Q timings are reported separately and combined.
Runs across multiple polynomial degrees D_POLY in {3, 5, 7, 9}.
"""

import os
import math
import time
import statistics
import numpy as np
import sys
sys.path.insert(0, "../..")
os.environ["OMP_NUM_THREADS"] = "1"

from vempc.crypto.setup import CryptoSetup
from vempc.crypto.offline import enc_matvec_ct_ct

# ------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------

RING_DIM   = 1 << 12
N_HOR      = 10
M_IN       = 2
DIM        = N_HOR * M_IN   # Nm = 20
P_CON      = 8               # number of constraints p
K          = 64              # samples per batch
N_TRIALS   = 20
D_POLY_SET = [3, 5, 7, 9]   # polynomial degrees to benchmark

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def rotation_sum(cc, ct, width):
    """Sum slots within each block of `width` via log2(width) EvalRotate+EvalAdd.
    Mirrors sum_within_slots in poly_eval.py."""
    acc  = ct
    step = 1
    while step < width:
        acc  = cc.EvalAdd(acc, cc.EvalRotate(acc, step))
        step *= 2
    return acc


def horner_eval(cc, ct, coeffs):
    """Slot-wise Horner evaluation of h_ell. Mirrors PolyEvaluator.horner_eval.
    Step 1: EvalMult(ct, pt)  [accumulator is a plaintext scalar].
    Steps 2..d: EvalMult(ct, ct)."""
    bs = cc.GetRingDimension() // 2

    def pt(v):
        return cc.MakeCKKSPackedPlaintext([float(v)] * bs)

    d      = len(coeffs) - 1
    acc_pt = float(coeffs[d])
    acc_ct = None

    for i in range(d - 1, -1, -1):
        if acc_ct is None:
            acc_ct = cc.EvalMult(ct, pt(acc_pt))   # EvalMult(ct, pt)
        else:
            acc_ct = cc.EvalMult(acc_ct, ct)        # EvalMult(ct, ct)
        acc_ct = cc.EvalAdd(acc_ct, pt(coeffs[i]))
    return acc_ct


def encrypt_vec(cc, keys, vec, bs):
    """Encrypt a real vector, zero-padded to bs slots."""
    padded = list(vec) + [0.0] * (bs - len(vec))
    return cc.Encrypt(keys.publicKey, cc.MakeCKKSPackedPlaintext(padded))


def encrypt_diags(cc, keys, M, bs):
    """Encrypt diagonal encoding of square matrix M (dim x dim).
    ct_diags[k] encrypts [M[i, (i+k) % dim] for i in range(dim)]."""
    dim = M.shape[0]
    cts = []
    for k in range(dim):
        diag   = [M[i, (i + k) % dim] for i in range(dim)]
        padded = diag + [0.0] * (bs - dim)
        cts.append(cc.Encrypt(keys.publicKey, cc.MakeCKKSPackedPlaintext(padded)))
    return cts


# ------------------------------------------------------------------
# Case 1: surrogate h_ell(g(U)) only  (tilted, tilde_kappa)
# ------------------------------------------------------------------

def case_1_surrogate(cc, ct_b, ct_Gamma_xi, coeffs):
    """
    Surrogate computation only.

    ct_g = ct_b + ct_{Gamma xi}       -- 1 EvalAdd
    ct_h = h_ell(ct_g)                -- Horner: 1 EvalMult(ct,pt) + (d-1) EvalMult(ct,ct)
    ct_s = rotation_sum(ct_h, P_CON)  -- log2(p) EvalRot + EvalAdd

    Returns ct_s: aggregated per-sample scores.
    """
    ct_g = cc.EvalAdd(ct_b, ct_Gamma_xi)
    ct_h = horner_eval(cc, ct_g, coeffs)
    ct_s = rotation_sum(cc, ct_h, P_CON)
    return ct_s


# ------------------------------------------------------------------
# Case 2: surrogate + J_Q(U)  (no tilting, kappa_0)
# ------------------------------------------------------------------

def case_2_surrogate(cc, ct_b, ct_Gamma_xi, coeffs):
    """Surrogate sub-block of Case 2. Identical to case_1_surrogate."""
    ct_g = cc.EvalAdd(ct_b, ct_Gamma_xi)
    ct_h = horner_eval(cc, ct_g, coeffs)
    ct_s = rotation_sum(cc, ct_h, P_CON)
    return ct_s


def case_2_J_Q(cc, ct_U, ct_STx0, H_diags_ct):
    """
    Quadratic cost J_Q(U) = (S^T x_0)^T U + (1/2) U^T H U.
    All operands are encrypted; all multiplications are EvalMult(ct, ct).

    Linear term (S^T x_0)^T U:
      ct_lin = rotation_sum( EvalMult(ct_{S^T x_0}, ct_U), DIM )
               -- 1 EvalMult(ct,ct) + log2(DIM) EvalRot + EvalAdd

    Quadratic term (1/2) U^T H U:
      ct_{HU}  = enc_matvec_ct_ct(H_diags, ct_U)
                 -- DIM EvalMult(ct,ct) + (DIM-1) EvalRot + EvalAdd
      ct_uHu   = EvalMult(ct_U, ct_{HU})
                 -- 1 EvalMult(ct,ct)
      ct_quad  = rotation_sum(ct_uHu, DIM)
                 -- log2(DIM) EvalRot + EvalAdd
      ct_quad  = EvalMult(ct_quad, pt_0.5)
                 -- 1 EvalMult(ct,pt)

    Returns ct_J = ct_lin + ct_quad.
    """
    bs = cc.GetRingDimension() // 2

    # Linear term
    ct_lin = rotation_sum(cc, cc.EvalMult(ct_STx0, ct_U), DIM)

    # Quadratic term
    ct_HU   = enc_matvec_ct_ct(cc, H_diags_ct, ct_U)
    ct_uHu  = cc.EvalMult(ct_U, ct_HU)
    ct_uHu  = rotation_sum(cc, ct_uHu, DIM)
    pt_half = cc.MakeCKKSPackedPlaintext([0.5] * bs)
    ct_quad = cc.EvalMult(ct_uHu, pt_half)

    return cc.EvalAdd(ct_lin, ct_quad)


# ------------------------------------------------------------------
# Main
# ------------------------------------------------------------------

def main():
    rot_set = set(range(1, DIM))
    rot_set.update(2**i for i in range(int(math.ceil(math.log2(DIM)))))
    rot_set.update(2**i for i in range(int(math.ceil(math.log2(P_CON)))))
    rot_amounts = sorted(rot_set)

    max_depth = max(D_POLY_SET) + 2

    print("Benchmark: Case 1 (surrogate) vs Case 2 (surrogate + J_Q)")
    print(f"  ring_dim={RING_DIM}, DIM={DIM}, p={P_CON}, K={K}, "
          f"trials={N_TRIALS}")
    print(f"  mult_depth={max_depth}, degrees={D_POLY_SET}")

    print("\n[1/3] Setting up crypto context ...")
    crypto = CryptoSetup(dim=DIM, k=K, ring_dim=RING_DIM,
                         mult_depth=max_depth, scaling_mod=50)
    crypto.setup()
    crypto.generate_rotation_keys(rot_amounts)
    cc  = crypto.cc
    bs  = crypto.batch_size

    # Synthetic private matrices for testing
    rng   = np.random.default_rng(42)
    A_raw = rng.standard_normal((DIM, DIM))
    H     = A_raw.T @ A_raw + np.eye(DIM)
    S_mat = rng.standard_normal((DIM, DIM))

    print("[2/3] Encrypting static quantities (offline phase) ...")
    H_diags_ct  = encrypt_diags(cc, crypto.keys, H, bs)
    ct_Gamma_xi = encrypt_vec(cc, crypto.keys, rng.standard_normal(P_CON), bs)

    print("[3/3] Running timing trials ...\n")

    header = (f"{'d_poly':>6}  "
              f"{'Case1 surr (ms)':>16}  "
              f"{'Case2 surr (ms)':>16}  "
              f"{'Case2 J_Q (ms)':>15}  "
              f"{'Case2 total (ms)':>17}  "
              f"{'Slowdown':>9}")
    print(header)
    print("-" * len(header))

    for d_poly in D_POLY_SET:
        coeffs = list(rng.standard_normal(d_poly + 1) * 0.3)

        t1_surr, t2_surr, t2_JQ = [], [], []

        for _ in range(N_TRIALS):
            U    = rng.standard_normal(DIM)
            STx0 = S_mat.T @ rng.standard_normal(DIM)
            b    = rng.standard_normal(P_CON)

            ct_U    = encrypt_vec(cc, crypto.keys, U,    bs)
            ct_STx0 = encrypt_vec(cc, crypto.keys, STx0, bs)
            ct_b    = encrypt_vec(cc, crypto.keys, b,    bs)

            # Case 1: surrogate only
            t0 = time.perf_counter()
            case_1_surrogate(cc, ct_b, ct_Gamma_xi, coeffs)
            t1_surr.append((time.perf_counter() - t0) * 1e3)

            # Case 2: surrogate sub-block timed separately
            t0 = time.perf_counter()
            case_2_surrogate(cc, ct_b, ct_Gamma_xi, coeffs)
            t2_surr.append((time.perf_counter() - t0) * 1e3)

            # Case 2: J_Q sub-block timed separately
            t0 = time.perf_counter()
            case_2_J_Q(cc, ct_U, ct_STx0, H_diags_ct)
            t2_JQ.append((time.perf_counter() - t0) * 1e3)

        med_1s  = statistics.median(t1_surr)
        med_2s  = statistics.median(t2_surr)
        med_2j  = statistics.median(t2_JQ)
        med_2t  = med_2s + med_2j
        slow    = med_2t / med_1s

        print(f"  d={d_poly}  "
              f"{med_1s:>16.2f}  "
              f"{med_2s:>16.2f}  "
              f"{med_2j:>15.2f}  "
              f"{med_2t:>17.2f}  "
              f"{slow:>8.2f}x")

    print()
    n_ctct_1 = "d-1"
    print("EvalMult(ct,ct) counts:")
    print(f"  Case 1 surrogate : d-1  (Horner steps 2..d)")
    print(f"  Case 2 surrogate : d-1  (same)")
    print(f"  Case 2 J_Q       : {DIM} (H*U diagonals) + 1 (U^T H U) "
          f"+ 1 (linear term) = {DIM+2}")
    print(f"  Case 2 total     : (d-1) + {DIM+2}")


if __name__ == "__main__":
    main()

Benchmark: Case 1 (surrogate) vs Case 2 (surrogate + J_Q)
  ring_dim=4096, DIM=20, p=8, K=64, trials=20
  mult_depth=11, degrees=[3, 5, 7, 9]

[1/3] Setting up crypto context ...
[2/3] Encrypting static quantities (offline phase) ...
[3/3] Running timing trials ...

d_poly   Case1 surr (ms)   Case2 surr (ms)   Case2 J_Q (ms)   Case2 total (ms)   Slowdown
-----------------------------------------------------------------------------------------
  d=3             89.03             89.04          1133.34            1222.38     13.73x
  d=5            120.26            120.28          1137.97            1258.25     10.46x
  d=7            145.44            145.43          1136.75            1282.19      8.82x
  d=9            150.46            150.53          1135.58            1286.11      8.55x

EvalMult(ct,ct) counts:
  Case 1 surrogate : d-1  (Horner steps 2..d)
  Case 2 surrogate : d-1  (same)
  Case 2 J_Q       : 20 (H*U diagonals) + 1 (U^T H U) + 1 (linear term) = 22
  Case 2 total  